# 22 — MultiIndex

MultiIndex (hierarchical indexing) enables powerful operations on multi-dimensional
data. It powers pivot_table results, groupby with multiple keys, and panel data.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Build a MultiIndex DataFrame directly
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Food', 'Books']
years = [2022, 2023]

idx = pd.MultiIndex.from_product(
    [regions, categories, years],
    names=['region', 'category', 'year']
)

sales = pd.DataFrame({
    'revenue': np.random.randint(100000, 1000000, len(idx)),
    'orders':  np.random.randint(50, 500, len(idx))
}, index=idx)

print(f"Shape: {sales.shape}")
print(sales.head(8))

## 1. Creating MultiIndex

In [ ]:
# from_tuples
mi1 = pd.MultiIndex.from_tuples(
    [('Eng', 'Junior'), ('Eng', 'Senior'), ('Sales', 'Junior'), ('Sales', 'Senior')],
    names=['department', 'level']
)
print(mi1)

# from_arrays
mi2 = pd.MultiIndex.from_arrays(
    [['Eng', 'Eng', 'Sales', 'Sales'],
     ['Junior', 'Senior', 'Junior', 'Senior']],
    names=['department', 'level']
)
print(mi2)

In [ ]:
# MultiIndex from groupby result
np.random.seed(42)
n = 500
raw = pd.DataFrame({
    'dept':   np.random.choice(['Eng', 'Sales', 'HR'], n),
    'level':  np.random.choice(['Junior', 'Senior'], n),
    'salary': np.random.randint(40000, 150000, n)
})

mi_from_groupby = raw.groupby(['dept', 'level'])['salary'].mean().round(0)
print(mi_from_groupby)
print(f"\nIndex type: {type(mi_from_groupby.index)}")

## 2. Indexing — loc with MultiIndex

In [ ]:
# Select one region
north = sales.loc['North']
print("North region:")
print(north)

In [ ]:
# Select region + category
north_electronics = sales.loc[('North', 'Electronics')]
print("North, Electronics:")
print(north_electronics)

In [ ]:
# Full tuple selection
north_electronics_2023 = sales.loc[('North', 'Electronics', 2023)]
print("North, Electronics, 2023:")
print(north_electronics_2023)

In [ ]:
# pd.IndexSlice — more readable cross-section
idx_slice = pd.IndexSlice

# Select North and South, all categories, 2023 only
subset = sales.loc[idx_slice[['North', 'South'], :, 2023], :]
print(f"North/South 2023: {subset.shape}")
print(subset)

## 3. xs() — Cross-Section

In [ ]:
# xs() selects a cross-section at any level
# Very useful for selecting from inner levels without specifying outer

# All Electronics rows (across all regions and years)
electronics_all = sales.xs('Electronics', level='category')
print("All Electronics (all regions, all years):")
print(electronics_all)

In [ ]:
# All 2023 rows
all_2023 = sales.xs(2023, level='year')
print("All 2023:")
print(all_2023)

## 4. swaplevel() and sort_index()

In [ ]:
# swaplevel() — reorder index levels
swapped = sales.swaplevel('region', 'category')
swapped = swapped.sort_index()
print("Swapped levels (category first):")
print(swapped.head(8))

In [ ]:
# sort_index() is REQUIRED after any index modification
# Otherwise loc/xs may raise PerformanceWarning or fail
sales_sorted = sales.sort_index()
print("Sorted index:")
print(sales_sorted.head(4))

## 5. stack() and unstack() with MultiIndex

In [ ]:
# unstack — move index level to column level
# unstack the 'year' level
revenue_wide = sales['revenue'].unstack('year')
print("Revenue unstacked by year:")
print(revenue_wide.head(8))

In [ ]:
# YoY calculation using unstacked form
revenue_wide['yoy_growth'] = (
    (revenue_wide[2023] - revenue_wide[2022]) / revenue_wide[2022] * 100
).round(1)
print(revenue_wide.head(8))

In [ ]:
# stack back to long form
back_to_long = revenue_wide.drop(columns='yoy_growth').stack('year')
back_to_long.name = 'revenue'
print(back_to_long.head(8))

## 6. reset_index() and set_index()

In [ ]:
# reset_index — convert index levels to columns (flattens MultiIndex)
flat = sales.reset_index()
print("Flat DataFrame:")
print(flat.head(5))
print(f"\nColumns: {flat.columns.tolist()}")

In [ ]:
# set_index — create MultiIndex from columns
rebuilt = flat.set_index(['region', 'category', 'year'])
print("Rebuilt MultiIndex:")
print(rebuilt.head(5))

In [ ]:
# reset_index(level=) — drop only specific levels
# Useful when you want to keep some index levels but flatten others
partial_reset = sales.reset_index(level='year')  # moves year to column
print(partial_reset.head(4))

## 7. GroupBy Producing MultiIndex

In [ ]:
# groupby with multiple keys → MultiIndex result
grp_result = raw.groupby(['dept', 'level']).agg(
    count=('salary', 'count'),
    avg_salary=('salary', 'mean')
).round(0)

print("MultiIndex from groupby:")
print(grp_result)

# Select specific group
print("\nEngineering only:")
print(grp_result.loc['Eng'])

## 8. Flattening MultiLevel Columns (from pivot_table)

In [ ]:
# pivot_table with multiple values creates MultiLevel columns
pt = pd.pivot_table(
    flat,
    values=['revenue', 'orders'],
    index='region',
    columns='year',
    aggfunc='sum'
)

print("MultiLevel columns from pivot_table:")
print(pt.columns)
print(pt)

In [ ]:
# Flatten column MultiIndex
pt.columns = [f'{col[0]}_{col[1]}' for col in pt.columns]
print("Flattened columns:")
print(pt)

## 9. Aggregation on MultiIndex

In [ ]:
# Sum within a level — groupby on specific index level
by_region = sales.groupby(level='region')['revenue'].sum()
print("Total revenue by region:")
print(by_region)

by_year = sales.groupby(level='year')['revenue'].sum()
print("\nTotal revenue by year:")
print(by_year)

In [ ]:
# mean across region — keeping category × year
by_cat_year = sales.groupby(level=['category', 'year'])['revenue'].mean().round(0)
print(by_cat_year)

## 10. MultiIndex Best Practices

| Best Practice | Why |
|---------------|-----|
| Always `sort_index()` after modification | Avoids PerformanceWarning, enables fast lookup |
| Use `pd.IndexSlice` for complex loc | Much more readable than nested tuples |
| Use `xs()` for inner-level selection | Cleaner than repeated loc chaining |
| Flatten with `reset_index()` before merge | merge requires flat columns |
| Flatten column MultiIndex after pivot_table | Avoid confusing nested column names |
| Use `unstack()` for comparison tables | Pivot-style display from groupby results |